In [1]:
import subprocess
import time
import json
import pandas as pd
from google.colab import drive
import os
#import ollama
drive.mount('/content/drive')

Mounted at /content/drive


Note: Im working inside google colab so i might need to rerun this from session to session

In [2]:
!pip install ollama
import ollama
!sudo apt-get install zstd -y

!curl -fsSL https://ollama.com/install.sh | sh

process = subprocess.Popen(['ollama', 'serve'],
                           stdout=subprocess.PIPE,
                           stderr=subprocess.PIPE)

# Give the server 15 seconds to wake up
time.sleep(15)

# 4. Pull the specific model (Llama 3.2 3B is very fast, 8B is smarter)
!ollama pull llama3.2:3b

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (569 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently i

In [8]:
def safe_romanian_extraction(description):
    system_prompt = """
    You are a Romanian Real Estate Expert.
    Read the provided description in Romanian.
    Extract the requested features.
    If the description does not mention a feature, use null.
    You're doing this to standardize data. Try to keep as much information as possible.
    In case you find a description from an agency, state clearly what's their commision.
    Do not guess. Only use information explicitly stated.
    """

    # We ask for the JSON keys in English for your programming ease,
    # but the LLM processes the Romanian text.
    prompt = f"Description to analyze: {description}"

    response = ollama.generate(model='llama3.2:3b', system=system_prompt, prompt=prompt)
    return response['response']

In [4]:
path_to_dataset =  os.path.join("/content/drive/MyDrive","Real estate parser")
olx_df = pd.read_csv(os.path.join(path_to_dataset,"Date OLX.csv"))
storia_df = pd.read_csv(os.path.join(path_to_dataset,"Date Storia Procesate.csv"))

In [9]:
raspuns = safe_romanian_extraction(storia_df.loc[0,'description'])

In [11]:
raspuns.split('\n')

['Baza de date standardizată a apartamentului:',
 '',
 '- Zona: Aparatorii Patriei - Metalurgiei',
 '- Durată de statie de metrou: 10 minute',
 '- Etaj: 1',
 '- Anul finalizării blocului: 2022',
 '- Suprafață: 48 mp',
 '- Mobilitate: Mobilat și utilat',
 '- Bucătărie: Dotată cu electrocasnicele necesare',
 '- Living: Luminos, echipat cu:',
 '\t+ Canapea',
 '\t+ TV',
 '\t+ AC',
 '- Dormitor: Cu pat matrimonial',
 '- Dressing: Incapator',
 '- Baie: 1',
 '- Parcare: Inclusă',
 '- Exclusiune: Animale de companie',
 '',
 'Comisie (dacă există): Nu este menționată în descriere']

In [13]:
storia_df.loc[0,'description'].split('\n')

['GLOBAL IMOBILIARE va propune spre inchiriere un apartament, mobilat si utilat modern in zona Aparatorii Patriei - Metalurgiei, la 10 minute de statia de metrou Aparatorii Patriei.',
 'Apartamentul este situat la etajul 1, blocul finalizat in anul 2022, avand o suprafata de 48 mp.',
 'Apartamentul este mobilat si utilat complet, iar bucataria este dotata cu toate electrocasnicele necesare.',
 'Dispune de un living luminos echipat cu o canapea, TV, AC, dormitor cu pat matrimonial, dressing incapator, 1 baie.',
 'EXCLUS ANIMALE DE COMPANIE!!!',
 'PARCARE INCLUSA!']